In [7]:
#Firts i will install the required libraries. 

%pip install sentence-transformers
%pip install  pypdf pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
# importing the libraries that i will use.

from sentence_transformers import SentenceTransformer # sentence-transformers is a library that allows me to use pretrained models to generate embeddings. 
import numpy as np
from pypdf import PdfReader
import pandas as pd
import re # re allow me to work with regular expressions 
from IPython.display import display #with this library i can display the dataframe better :). 

In [16]:
# Now i will load the model to generate emeddings. 
model = SentenceTransformer("intfloat/multilingual-e5-base")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

... (2 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [19]:
print("Máximo de tokens:", model.max_seq_length)

Máximo de tokens: 512


In [20]:
# i will read the pdf  of the code of conduct of the Javveriana University. 

reader = PdfReader("CodeOfConductJaverianaUniversity.pdf")

print("Number of pages in the PDF:", len(reader.pages))


Number of pages in the PDF: 32


In [21]:
# Extracting text from each page and storing it in a list.
 
pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pages.append({
        "page": page_number,
        "text": text
    })




In [22]:
# Initialize the tokenizer (tokenisador del modelo) 

tokenizer = model.tokenizer

In [23]:
# i wiil create a list of chunks, where each chunk is a dictionary containing the page number and the corresponding text. 
# I will also tokenize the text and print the number of tokens for each page.

chunks = []
for page_data in pages:
    page_number = page_data["page"]
    text = page_data["text"]

    tokens = tokenizer.encode(text, add_special_tokens=False)
    #print("Page:", page_number)
    #print("Number of tokens:", len(tokens))
    if len(tokens) > 512:
        print("Warning: Page", page_number, "exceeds the maximum token limit of 512.")




[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (598 > 512). Running this sequence through the model will result in indexing errors


In [24]:
# Now i will split the text by articles
# Articles longer than the model limit (512 tokens) are split into sub-chunks with overlap.

FIRST_PAGE_REGLAMENTO = 3 #  Reglamento starts on page 3
MAX_TOKENS = 450    # leave room for the "passage: " prefix and the article header
OVERLAP = 50

#^means the start of the line, \s* means any number of whitespace characters 
# (\d{1,3}) means a group of 1 to 3 digits, \. means a dot, 
# and \s means a whitespace character.
article_pattern = re.compile(r"^\s*(\d{1,3})\.\s") 
#A-Z uppercase letters, ÁÉÍÓÚÑ uppercase accented letters, 0-9 digits, space, comma,
heading_pattern = re.compile(r"^[A-ZÁÉÍÓÚÑ0-9 ,\-]+$")      

articles = []
current = None
next_number = 1   #The next is the first article. 
section = ""

for page_data in pages:

    if page_data["page"] < FIRST_PAGE_REGLAMENTO:
        continue

    for line in page_data["text"].split("\n"):

        line = line.strip() # with this i remove the whitespaces. 
        if not line:
            continue

        match = article_pattern.match(line) #i verify if the line is consisten with the pattern. 

        if match and int(match.group(1)) == next_number:
            if current is not None:
                articles.append(current)
            current = {
                "article": next_number,
                "page": page_data["page"],
                "section": section,
                "text": line
            }
            next_number += 1

        elif heading_pattern.match(line) and len(line) > 3:
            section = line          # section title, not part of the article text

        elif current is not None:
            current["text"] += " " + line

if current is not None:
    articles.append(current)

print("Number of articles:", len(articles))

# Building the chunks each one per article or many if the article is too long

chunks = []

for art in articles:

    header = f"Artículo {art['article']} ({art['section']}): "
    tokens = tokenizer.encode(art["text"], add_special_tokens=False)

    if len(tokens) <= MAX_TOKENS:
        chunks.append({
            "article": art["article"],
            "page": art["page"],
            "text": header + art["text"]
        })
        continue

    start = 0
    while start < len(tokens):
        piece = tokenizer.decode(tokens[start:start + MAX_TOKENS], skip_special_tokens=True)
        chunks.append({
            "article": art["article"],
            "page": art["page"],
            "text": header + piece
        })
        if start + MAX_TOKENS >= len(tokens):
            break
        start += MAX_TOKENS - OVERLAP

print("Number of chunks:", len(chunks))

Number of articles: 158
Number of chunks: 167


In [25]:
# Now i will print the number of tokens for each chunk to verify that the chunks are within the specified token limit.

for chunk in chunks:
    chunk_tokens = tokenizer.encode(chunk["text"], add_special_tokens=False)
    print("Article:", chunk["article"], "Page:", chunk["page"], "Tokens:", len(chunk_tokens))

Article: 1 Page: 3 Tokens: 76
Article: 2 Page: 3 Tokens: 35
Article: 3 Page: 3 Tokens: 72
Article: 4 Page: 3 Tokens: 62
Article: 5 Page: 3 Tokens: 65
Article: 6 Page: 3 Tokens: 43
Article: 7 Page: 3 Tokens: 424
Article: 8 Page: 4 Tokens: 461
Article: 8 Page: 4 Tokens: 203
Article: 9 Page: 5 Tokens: 99
Article: 10 Page: 6 Tokens: 34
Article: 11 Page: 6 Tokens: 91
Article: 12 Page: 6 Tokens: 88
Article: 13 Page: 6 Tokens: 52
Article: 14 Page: 6 Tokens: 100
Article: 15 Page: 6 Tokens: 51
Article: 16 Page: 6 Tokens: 89
Article: 17 Page: 6 Tokens: 125
Article: 18 Page: 7 Tokens: 117
Article: 19 Page: 7 Tokens: 112
Article: 20 Page: 7 Tokens: 78
Article: 21 Page: 7 Tokens: 39
Article: 22 Page: 7 Tokens: 47
Article: 23 Page: 7 Tokens: 76
Article: 24 Page: 7 Tokens: 55
Article: 25 Page: 7 Tokens: 72
Article: 26 Page: 7 Tokens: 76
Article: 27 Page: 7 Tokens: 76
Article: 28 Page: 8 Tokens: 59
Article: 29 Page: 8 Tokens: 66
Article: 30 Page: 8 Tokens: 114
Article: 31 Page: 8 Tokens: 63
Article: 3

In [11]:
# Now i will prepare the text for the embedding model. 

texts_for_embedding = []

for chunk in chunks:

    text = chunk["text"]

    text_for_model = "passage: " + text

    texts_for_embedding.append(text_for_model)
    # Now i will generate an embedding vector for each chunk

embeddings = model.encode(
    texts_for_embedding,
    convert_to_numpy=True
)



In [26]:
#  query.

query = input("Enter your question:")

query_for_model = "query: " + query

query_embedding = model.encode(
    query_for_model,
    convert_to_numpy=True
)

# and i calculate de norm of the query emebedding vector. 

query_norm = np.linalg.norm(query_embedding)
print(query)

que pasa si un estudiante se copia e un parcial. 


In the following code, I calculate the cosine similarity between the query vector and the text vector using:

$$
\cos(\theta) =
\frac{\mathbf{q}^{T}\mathbf{t}}
{\|\mathbf{q}\|\,\|\mathbf{t}\|}
$$

The angle between these vectors is:

$$
\theta = \arccos\left(
\frac{\mathbf{q}^{T}\mathbf{t}}
{\|\mathbf{q}\|\,\|\mathbf{t}\|}
\right)
$$

where $\mathbf{q}$ is the query embedding vector, $\mathbf{t}$ is the text embedding vector, $\mathbf{q}^{T}\mathbf{t}$ is their dot product, and $\|\mathbf{q}\|$ and $\|\mathbf{t}\|$ are their Euclidean norms. 

In [ ]:

# Compare the query vector with every chunk embedding

results = []

for i in range(len(embeddings)):

    chunk_embedding = embeddings[i]
    chunk_embedding = np.asarray(embeddings[i], dtype=np.float32).reshape(-1)
    query_embedding = np.asarray(query_embedding, dtype=np.float32).reshape(-1)
    dot_product = np.dot(
        query_embedding,
        chunk_embedding
    )

    chunk_norm = np.linalg.norm(
        chunk_embedding
    )

    cosine_similarity = dot_product / (
        query_norm * chunk_norm
    )
    theta = np.arccos(cosine_similarity)

    

    results.append({
        "chunk": i + 1,
        "page": chunks[i]["page"],
         "article": chunks[i]["article"],
        "cosine_similarity": cosine_similarity,
        "Angle between vectors (radians)": theta,
        "text": chunks[i]["text"]
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="cosine_similarity",
    ascending=False
)

ValueError: shapes (768,) and (384,) not aligned: 768 (dim 0) != 384 (dim 0)

In [14]:
TOP_K = 5

top_results = results_df.head(TOP_K)

print(top_results)

     chunk  page  article  cosine_similarity  Angle between vectors (radians)  \
134    135    26      127           0.863068                         0.529485   
160    161    31      152           0.861577                         0.532428   
120    121    20      120           0.859949                         0.535626   
80      81    14       80           0.855977                         0.543359   
125    126    22      121           0.855725                         0.543847   

                                                  text  
134  Artículo 127 (SANCIONES DISCIPLINARIAS): 127. ...  
160  Artículo 152 (OTRAS DISPOSICIONES): 152. De to...  
120  Artículo 120 (FALTAS DISCIPLINARIAS): 120. De ...  
80   Artículo 80 (VALIDACIÓN DE ASIGNATURAS): 80. E...  
125  Artículo 121 (FALTAS DISCIPLINARIAS): tercero,...  


In [15]:

tabla_final = top_results[
    ["article", "page", "Angle between vectors (radians)", "cosine_similarity", "text"]
].copy()

tabla_final.insert(0, "rank", range(1, len(tabla_final) + 1))
tabla_final = tabla_final.rename(columns={
    "rank": "Puesto",
    "article": "Artículo",
    "page": "Página",
    "Angle between vectors (radians)": "Ángulo entre vectores (radianes)",
    "cosine_similarity": "Similitud",
    "text": "Texto"
})

display(
    tabla_final.style
    .hide(axis="index")
    .format({"Similitud": "{:.3f}"})
    .bar(subset=["Similitud"], color="#72b7b2")
    .bar(subset=["Ángulo entre vectores (radianes)"], color="#dc7622")
    .set_caption(f"Consulta: {query}")
)

Puesto,Artículo,Página,Ángulo entre vectores (radianes),Similitud,Texto
1,127,26,0.529485,0.863,"Artículo 127 (SANCIONES DISCIPLINARIAS): 127. Una vez se determine el tipo de falta, la autoridad competente podrá calificar la gravedad de la sanción de conformidad con las siguientes circunstancias de atenuación o agravación de la conducta. a. Circunstancias de atenuación: Se entenderán como circunstancias de atenuación de la conducta, las siguientes: a.1. Aceptar la comisión de la falta disciplinaria antes del fallo de primera instancia. a.2. Evitar la injusta vinculación de terceros al proceso. a.3. Suministrar información que sirva para esclarecer los hechos o que permita establecer la participación de otras personas en la conducta que se investiga. a.4. No contar con antecedentes disciplinarios. b. Circunstancias de agravación: Se entenderán como circunstancias de agravación de la conducta, las siguientes: b.1. Tener una sanción disciplinaria previa. b.2. Realizar el hecho en complicidad o coautoría con otros estudiantes, empleados de la Universidad o terceros. b.3. Adelantar actos dirigidos a ocultar las pruebas y/o evitar que los testigos rindan su declaración. b.4. Ejercer algún tipo de presión contra las autoridades disciplinarias competentes. b.5. Que la conducta sea llevada a cabo por un estudiante miembro de un órgano colegiado o comité constituido reglamentariamente por la Universidad. b.6. Que el estudiante tenga alguna vinculación laboral y/o civil con la Universidad. b.7. Causar daños a las instalaciones o bienes de la Universidad con ocasión de la comisión de una falta disciplinaria. b.8. Que la conducta cometida haya tenido como motivación la obtención de un beneficio económico o patrimonial. b.9. Aquellas conductas que, debido a su naturaleza, intención lesiva y gravedad del daño, constituyan un grave atentado en contra del buen nombre de la Universidad y del bienestar general de la Comunidad Educativa Javeriana."
2,152,31,0.532428,0.862,"Artículo 152 (OTRAS DISPOSICIONES): 152. De toda comunicación que se envíe al estudiante en el desarrollo de estos procesos se debe conservar copia en el expediente del estudiante. El estudiante firmará constancia de que la recibió y se anotará la fecha en que lo hizo. Si se niega a firmar, se le hará entrega de la comunicación delante de dos testigos quienes dejarán constancia escrita de ello. No siendo posible localizar al estudiante para entregarle personalmente la comunicación, o si este se negare a recibirla, se le remitirá por correo certificado y electrónico a las direcciones registradas en la Universidad. En este caso, los términos que correspondan comenzarán a correr a partir del día siguiente a la fecha de envío y el proceso se resolverá con las pruebas que reposen en el mismo."
3,120,20,0.535626,0.860,"Artículo 120 (FALTAS DISCIPLINARIAS): 120. De conformidad con el ámbito de aplicación previsto en el numeral 118 constituyen faltas graves, entre otras: a. El fraude en cualquier clase de actividad, trabajo y evaluación académica; o el incumplimiento de las reglas definidas por la Universidad, o de las instrucciones impartidas previamente por un profesor, para el desa rrollo de exámenes, evaluaciones, parciales, trabajos, talleres, historias clínicas, prác ticas, rotaciones, entre otros. Se considera fraude académico, entre otras, las siguientes conductas: a.1. Incurrir en copia, de manera total o parcial en evaluaciones académicas, en documentos o en trabajos académicos. Esta falta cobija tanto al estudiante que la realiza como al estudiante que voluntariamente la permite. a.2. La posesión o utilización de material o ayudas no autorizadas previamente por el profesor durante los exámenes o pruebas académicas, tales como: anotaciones, apuntes, fórmulas, celulares, tabletas o cualquier instrum ento o dispositivo electrónico. a.3. Establecer comunicación no autorizada con cualquier persona y por cualquier medio (verbal, digital o gestual), durante el desarrollo de una eva